In [149]:
import torch
import torch.nn.functional as F
from math import ceil, floor

In [180]:
def irfft(y_real: torch.Tensor, y_imag: torch.Tensor, n_fft: int) -> torch.Tensor:
    assert y_real.shape == y_imag.shape, "Real and imaginary parts must have the same shape"
    assert n_fft == y_real.shape[-1], "n_fft must be equal to the length of the real part"
    arg_matrix = (2.0 * torch.pi / n_fft) * (
        torch.arange(n_fft, dtype=torch.float32)[:, None] * torch.arange(n_fft, dtype=torch.float32).float()[None, :]
    )
    cos_matrix = torch.cos(arg_matrix)
    sin_matrix = torch.sin(arg_matrix)
    x = (torch.matmul(cos_matrix, y_real.transpose(0, -1)) - torch.matmul(sin_matrix, y_imag.transpose(0, -1))) / n_fft
    return x.transpose(0, -1)


def extend_input(y_real: torch.Tensor, y_imag: torch.Tensor, n_fft: int) -> tuple[torch.Tensor, torch.Tensor]:
    assert y_real.shape == y_imag.shape, "Real and imaginary parts must have the same shape"
    shape = y_real.shape
    n = shape[-1]
    m = 2 * n - 2
    r = n_fft - m
    real = []
    imag = []
    if r > 0:
        i = n
        j = n
        s = r - 1
    else:
        i = n + (r // 2)
        j = n_fft - i + 1
        s = 0
    real.append(y_real[..., :i])
    imag.append(y_imag[..., :i])
    if s > 0:
        real.append(torch.zeros((*shape[:-1], s)))
        imag.append(torch.zeros((*shape[:-1], s)))
    real.append(y_real[..., 1:j].flip(-1))
    imag.append(-y_imag[..., 1:j].flip(-1))

    return torch.concatenate(real, dim=-1), torch.concatenate(imag, dim=-1)

In [183]:
# x = F.pad(torch.arange(1, 101, dtype=torch.float32), (0, 1), mode='constant', value=0.0).unsqueeze(0)
x = torch.randn(16, 1000)  # Example input tensor
y = torch.fft.rfft(x)
print(f"x = {x}")
print(f"y = {y}")

x = tensor([[-0.5380,  1.6538,  1.6759,  ..., -0.4333, -0.2211,  0.1308],
        [ 1.3351, -1.3815, -1.0603,  ...,  1.8284,  0.3951, -0.4860],
        [ 0.2304, -3.0830, -0.9937,  ...,  0.1075,  0.6734,  1.0013],
        ...,
        [-0.7859,  0.8443,  0.2656,  ...,  0.2416, -1.3029, -0.7692],
        [ 0.8361, -1.4575,  1.4408,  ..., -1.3934, -0.8754,  0.1764],
        [ 0.0969, -0.5423,  0.2181,  ..., -0.3448,  0.6313,  0.3712]])
y = tensor([[ 27.0368+0.0000j,   4.7378-43.6413j,  -7.0519+6.6705j,
          ..., -33.8889+20.5838j,   3.1110-3.6065j,
          23.0168+0.0000j],
        [ -6.5063+0.0000j, -41.0268-66.9113j,  29.6857-22.9849j,
          ...,  35.2072-26.7587j,   4.1822+15.0961j,
          -7.9352+0.0000j],
        [-23.8832+0.0000j, -24.4476+10.0221j,  -0.0606-15.0723j,
          ...,  22.6497-6.6451j,  45.0002-41.7445j,
          -6.7577+0.0000j],
        ...,
        [ 23.4257+0.0000j,   1.4670-6.1889j, -10.9698+38.1813j,
          ..., -19.0849-30.4824j,  -3.9440+69.

In [184]:
x1 = torch.fft.irfft(y, n=100)
print(f"x1 = {x1}")

x1 = tensor([[ 4.1486,  1.6699, -1.7658,  ...,  4.0341, -1.5688, -0.6895],
        [-0.1133, -5.5614,  4.0890,  ..., -5.6030,  1.9976, -3.4170],
        [-5.8554,  0.9415,  2.8701,  ..., -2.4903, -1.9839, -8.8889],
        ...,
        [-2.9966,  8.0610, -0.2039,  ...,  1.0647,  1.4895,  1.4499],
        [-1.3575,  0.0661, -5.2099,  ..., -3.7724,  5.4403,  2.1126],
        [-1.4142, -1.8095,  2.4638,  ..., -1.7220, -0.7566, -4.5655]])


In [185]:
y_real_padded, y_imag_padded = extend_input(torch.real(y), torch.imag(y), 100)
print(f"y_real_padded = {y_real_padded}")
print(f"y_imag_padded = {y_imag_padded}")

y_real_padded = tensor([[ 27.0368,   4.7378,  -7.0519,  ...,  31.3631,  -7.0519,   4.7378],
        [ -6.5063, -41.0268,  29.6857,  ..., -15.7941,  29.6857, -41.0268],
        [-23.8832, -24.4476,  -0.0606,  ...,  -3.9984,  -0.0606, -24.4476],
        ...,
        [ 23.4257,   1.4670, -10.9698,  ...,  13.2274, -10.9698,   1.4670],
        [ -3.0064, -32.1683,   7.7877,  ..., -31.0111,   7.7877, -32.1683],
        [-16.3980, -11.6131, -25.5069,  ...,  36.0133, -25.5069, -11.6131]])
y_imag_padded = tensor([[  0.0000, -43.6413,   6.6705,  ...,  31.0580,  -6.6705,  43.6413],
        [  0.0000, -66.9113, -22.9849,  ...,  -2.4598,  22.9849,  66.9113],
        [  0.0000,  10.0221, -15.0723,  ...,  11.6973,  15.0723, -10.0221],
        ...,
        [  0.0000,  -6.1889,  38.1813,  ..., -16.9393, -38.1813,   6.1889],
        [  0.0000,   8.3798,  15.0919,  ...,  20.2485, -15.0919,  -8.3798],
        [  0.0000,   8.5264,  -1.5818,  ...,  17.3170,   1.5818,  -8.5264]])


In [186]:
x2 = irfft(y_real_padded, y_imag_padded, 100)
print(f"x2 = {x2}")

x2 = tensor([[ 4.1486,  1.6699, -1.7658,  ...,  4.0341, -1.5687, -0.6895],
        [-0.1133, -5.5614,  4.0890,  ..., -5.6031,  1.9976, -3.4170],
        [-5.8554,  0.9415,  2.8701,  ..., -2.4903, -1.9839, -8.8889],
        ...,
        [-2.9966,  8.0610, -0.2039,  ...,  1.0647,  1.4895,  1.4499],
        [-1.3575,  0.0661, -5.2099,  ..., -3.7724,  5.4403,  2.1126],
        [-1.4142, -1.8095,  2.4638,  ..., -1.7219, -0.7566, -4.5655]])


In [187]:
torch.allclose(x1, x2)  # Should be True if the implementation is correct
print(f"Are x1 and x2 close? {torch.allclose(x1, x2, atol=1e-3)}")
print(torch.abs(x1 - x2).max())  # Should be very small if the implementation is correct

Are x1 and x2 close? True
tensor(9.8109e-05)
